# Task 3 – Running a model locally with Ollama: 16-bit vs 4-bit

**Model:** Qwen2.5-3B-Instruct, run twice by Ollama: **fp16** (16-bit weights, about 6.2 GB) and **q4_K_M** (4-bit weights, about 1.9 GB). Everything else is identical: same 10 prompts, greedy decoding (temperature 0), 4096-token context.
**Run:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all* (allow 10 to 15 minutes: about 8 GB of models are downloaded). On a CPU runtime it also works with the 0.5B model (see cell 2), only slower.

| The task asks for | Where (code cells are numbered `# 1` to `# 5`) |
|---|---|
| Tokens per second and memory for each setting | cell 3 (table) |
| Quality on the same 10 prompts, with observations | cell 3 (rough auto-check) and cell 4 (answers side by side) |
| KV cache and why quantisation saves memory | cell 5 (numbers) and the drafts below |
| Plan for 20 users on one 24 GB GPU | last section |

In [1]:
# 1 - install Ollama and start its server (GPU runtime)
!apt-get -qq update && apt-get -qq install -y zstd pciutils lshw
!curl -fsSL https://ollama.com/install.sh | sh

import re, shutil, subprocess, time, requests
import pandas as pd

assert shutil.which("ollama"), "Ollama did not install: run this cell again"
HOST = "http://127.0.0.1:11434"

def up():
    try: return requests.get(HOST, timeout=2).ok
    except requests.RequestException: return False

if not up():
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(60):
        if up(): break
        time.sleep(1)
assert up(), "the Ollama server did not start"

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package pci.ids.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../0-pci.ids_0.0~2024.03.31-1ubuntu0.1_all.deb ...
Unpacking pci.ids (0.0~2024.03.31-1ubuntu0.1) ...
Selecting previously unselected package libpci3:amd64.
Preparing to unpack .../1-libpci3_1%3a3.10.0-2build1_amd64.deb ...
Unpacking libpci3:amd64 (1:3.10.0-2build1) ...
Selecting previously unselected package lshw.
Preparing to unpack .../2-lshw_02.19.git.2021.06.19.996aaad9c7-2ubuntu0.24.04.1_amd64.deb ...
Unpacking lshw (02.19.git.2021.06.19.996aaad9c7-2ubuntu0.24.04.1) ...
Selecting previously unselected package pciutils.
Preparing to unpack .../3-pciutils_1%3a3.10.0-2build1_amd64.deb ...
Unpacking pciutils (1:3.10.0-2build1) ...
Selecting previously unselec

### Note on Systemd Warning
The warning `WARNING: systemd is not running` is normal and expected inside a Google Colab notebook. Colab runtimes do not use `systemd`. Ollama is successfully running in the background as a subprocess initiated by python in cell 1.

In [2]:
# 2 - the two settings: the same model with 16-bit and with 4-bit weights, then download both
BASE = "qwen2.5:3b-instruct"                                              # on a CPU runtime use "qwen2.5:0.5b-instruct"
SETTINGS = {"fp16": BASE + "-fp16", "q4_K_M": BASE + "-q4_K_M"}           # any quantisation level in the Ollama library works, e.g. "-q8_0" or "-q2_K"
A, B = SETTINGS.values()
!ollama pull $A
!ollama pull $B

## Speed, memory and quality
For each setting, the same 10 prompts are sent one after another. **Speed** is the average number of generated tokens per second (`eval_count / eval_duration` reported by Ollama; model loading is not counted). **Memory** is what the Ollama server reports for the loaded model (`/api/ps`): the weights plus the KV cache for a 4096-token context plus working buffers (`nvidia-smi` shows a bit more, because of CUDA's own overhead). The **rough auto-check** only tests whether the answer contains the expected word or number, so it is a quick signal and not a grade; judge the quality yourself from the side-by-side answers in cell 4.

In [3]:
# 3 - run the same 10 prompts on both settings; record speed and memory
PROMPTS = [     # (prompt, regex a correct answer must match, or None when the answer is open-ended)
    ("What is 17 * 23? Answer with just the number.", r"\b391\b"),
    ("What is the capital of Australia? Answer in one word.", r"canberra"),
    ("A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost? Think step by step, then answer.", r"(?<!\d)0?\.05\b|\b5 cents|five cents"),
    ("A train travels 60 km in 45 minutes. What is its average speed in km/h? Answer with just the number.", r"\b80\b"),
    ("If all Bloops are Razzies and all Razzies are Lazzies, are all Bloops definitely Lazzies? Start with yes or no, then explain in one sentence.", r"^\W*yes"),
    ("Translate to French: 'Where is the train station?'", r"gare"),
    ("Write a Python function fib(n) that returns the n-th Fibonacci number. Reply with code only.", r"def fib.*return"),
    ("Explain in two sentences why the sky is blue.", r"scatter"),
    ("Summarise in one sentence: The mitochondrion is an organelle found in most eukaryotic cells. It produces most of the cell's ATP through oxidative phosphorylation, which is why it is often called the powerhouse of the cell.", r"ATP|energy|powerhouse"),
    ("Write a polite two-sentence email declining a meeting invitation.", None),
]

def generate(tag, prompt, n):
    r = requests.post(f"{HOST}/api/generate", json={"model": tag, "prompt": prompt, "stream": False,
                      "options": {"temperature": 0, "seed": 0, "num_predict": n, "num_ctx": 4096}})
    r.raise_for_status()
    return r.json()

def run(tag):
    generate(tag, "Hello", 8)                                             # warm-up: loads the model, so loading time is not measured
    results = [generate(tag, p, 300) for p, _ in PROMPTS]
    ps = next(m for m in requests.get(f"{HOST}/api/ps").json()["models"] if m["model"] == tag)    # memory the server holds for this model right now
    requests.post(f"{HOST}/api/generate", json={"model": tag, "keep_alive": 0})                  # unload it, so the next setting is measured alone
    return results, ps

runs = {name: run(tag) for name, tag in SETTINGS.items()}
disk = {m["name"]: m["size"] for m in requests.get(f"{HOST}/api/tags").json()["models"]}
check = lambda out, rx: bool(rx and re.search(rx, out, re.I | re.S))
n_checks = sum(rx is not None for _, rx in PROMPTS)

pd.DataFrame({name: {
    "file size on disk (GB)": disk[SETTINGS[name]] / 1e9,
    "memory held by Ollama (GB)": ps["size"] / 1e9,
    "   of which on the GPU (GB)": ps["size_vram"] / 1e9,
    "speed (generated tokens/s)": sum(r["eval_count"] for r in res) / (sum(r["eval_duration"] for r in res) / 1e9),
    f"rough auto-check (out of {n_checks})": sum(check(r["response"], rx) for r, (_, rx) in zip(res, PROMPTS)),
} for name, (res, ps) in runs.items()}).round(2)

,fp16,q4_K_M
file size on disk (GB),6.18,1.93
memory held by Ollama (GB),6.41,2.16
of which on the GPU (GB),6.41,2.16
speed (generated tokens/s),38.66,83.09
rough auto-check (out of 9),8.00,8.00


In [4]:
# 4 - the answers side by side:  OK = contains the expected answer, -- = does not, blank = open-ended prompt
names = list(SETTINGS)
text = lambda name, i: runs[name][0][i]["response"].strip()
print(f"identical answers in both settings: {sum(text(names[0], i) == text(names[1], i) for i in range(len(PROMPTS)))} of {len(PROMPTS)}\n")
for i, (p, rx) in enumerate(PROMPTS):
    print(f"[{i + 1}] {p}")
    for name in names:
        print(f"   {name:>7} {'OK ' if check(text(name, i), rx) else '-- ' if rx else '   '}{text(name, i).replace(chr(10), ' ⏎ ')[:400]}")
    print()

identical answers in both settings: 4 of 10

[1] What is 17 * 23? Answer with just the number.
      fp16 OK 391
    q4_K_M OK 391

[2] What is the capital of Australia? Answer in one word.
      fp16 OK Canberra
    q4_K_M OK Canberra

[3] A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost? Think step by step, then answer.
      fp16 OK Let's solve this problem step by step. ⏎  ⏎ 1. **Define the variables:** ⏎    - Let \( B \) be the cost of the ball in dollars. ⏎    - Let \( T \) be the cost of the bat in dollars. ⏎  ⏎ 2. **Set up the equations based on the given information:** ⏎    - The total cost of the bat and the ball is $1.10. ⏎      \[ ⏎      T + B = 1.10 ⏎      \] ⏎    - The bat costs $1.00 more than the ball. ⏎      \[ 
    q4_K_M OK Let's solve this problem step by step. ⏎  ⏎ 1. Let's denote the cost of the ball as \( x \) dollars. ⏎ 2. According to the problem, the bat costs $1.00 more than the ball. Therefore, the cos

In [6]:
import pandas as pd

# Retrieve current metrics dynamically from your run results
summary_data = {}
for name, (res, ps) in runs.items():
    total_tokens = sum(r["eval_count"] for r in res)
    total_duration_ns = sum(r["eval_duration"] for r in res)
    speed_tps = total_tokens / (total_duration_ns / 1e9) if total_duration_ns > 0 else 0

    correct_count = sum(check(r["response"], rx) for r, (_, rx) in zip(res, PROMPTS))

    summary_data[name] = {
        "File Size (GB)": disk[SETTINGS[name]] / 1e9,
        "Ollama RAM/VRAM Memory (GB)": ps["size"] / 1e9,
        "Generation Speed (tokens/s)": speed_tps,
        "Auto-check Score (out of 9)": f"{correct_count}/9"
    }

df_comparison = pd.DataFrame(summary_data).round(2)
display(df_comparison)

,fp16,q4_K_M
File Size (GB),6.178327,1.929912
Ollama RAM/VRAM Memory (GB),6.411969,2.163569
Generation Speed (tokens/s),38.656149,83.088805
Auto-check Score (out of 9),8/9,8/9


**What to look at for your observations:** the auto-check score and the number of identical answers; where the two answers differ (length, wording, a wrong number, a reasoning slip); and which prompts are most sensitive (arithmetic and multi-step word problems tend to be). Speed: decoding reads all weights once per generated token, so it is limited by memory bandwidth, and a file about 3.3× smaller can generate up to about 3× more tokens per second.

In [5]:
# 5 - how big is the KV cache for this model?
info = requests.post(f"{HOST}/api/show", json={"model": SETTINGS["fp16"]}).json()["model_info"]
arch = info["general.architecture"]
layers, kv_heads, heads, emb = (info[f"{arch}.{k}"] for k in ("block_count", "attention.head_count_kv", "attention.head_count", "embedding_length"))
head_dim = info.get(f"{arch}.attention.key_length", emb // heads)
per_token = 2 * layers * kv_heads * head_dim * 2                          # keys and values (2) x 2 bytes each (fp16)
print(f"{layers} layers x {kv_heads} KV heads x {head_dim} values per head  ->  {per_token / 1024:.0f} KiB of KV cache per token")
print(f"one 4096-token chat: {per_token * 4096 / 1e6:.0f} MB | 20 users x 4096 tokens: {20 * per_token * 4096 / 1e9:.1f} GB"
      f" | the fp16 weights: {disk[SETTINGS['fp16']] / 1e9:.1f} GB")

36 layers x 2 KV heads x 128 values per head  ->  36 KiB of KV cache per token
one 4096-token chat: 151 MB | 20 users x 4096 tokens: 3.0 GB | the fp16 weights: 6.2 GB


## Drafts for the written parts (rewrite them in your own words, and use your own numbers)

**What the KV cache is.** To produce each new token, attention needs the *keys* and *values* of every earlier token, in every layer. Recomputing them at every step would make the total cost of generating a text grow with the square of its length instead of in proportion to it, so each token's keys and values are computed once, stored, and reused: that store is the KV cache. A new token only computes its own key and value and then attends over the stored ones. The price is memory, which grows linearly with the context length and with the number of simultaneous users: `2 (keys and values) × layers × KV heads × head size × bytes per number` per token (cell 5). It is small here (grouped-query attention gives this model only 2 KV heads), but for a 7–8B model with long chats or many users it can rival the weights.

**Why quantisation saves memory.** Weights are stored as numbers. In fp16 each takes 16 bits. 4-bit quantisation stores each weight as a small integer (16 possible levels) plus one scale value shared by a small group of weights (typically 32 to 128), which turns the integers back into approximate real numbers while the model runs. That is about 4.5 to 5 bits per weight instead of 16, so the file and the memory shrink by roughly 3× (the table in cell 3 shows the exact ratio for this model). The rounding error is spread over billions of weights and the per-block scale keeps the range right, so quality drops only a little at 4 bits; it drops much faster at 2 to 3 bits. Fewer bytes to read per token is also why the 4-bit model generates faster.

## A back-of-envelope plan: 20 users at the same time on one 24 GB GPU
**Assumptions.** One 24 GB card (L4: 300 GB/s, A10G: 600 GB/s, RTX 4090: about 1,000 GB/s). An 8B-class chat model with Llama-3.1-8B's shape (32 layers, 8 KV heads, head size 128). Average conversation of 2,000 tokens per user, capped at 4,096. All 20 users generating at the same moment (the worst case; in reality many are reading or typing). The serving software keeps 10% of the memory free and needs about 2 GB for the CUDA context and buffers. Decoding reaches about 60% of the peak memory bandwidth. Prefill reaches about 40 TFLOP/s.

**Step 1 – KV cache per token (fp16):** 2 × 32 layers × 8 KV heads × 128 × 2 bytes = 131,072 bytes = **128 KiB**.

**Step 2 – memory budget.** Usable memory is 0.9 × 24 = 21.6 GB; minus 2 GB overhead leaves 19.6 GB for weights and KV cache.

| | Weights | Left for KV cache | Tokens that fit (÷ 128 KiB) | Users at 4,096 tokens | Users at 2,000 tokens |
|---|---|---|---|---|---|
| fp16 (8B × 2 bytes) | 16 GB | 3.6 GB | 27,000 | **6** | 13 |
| 4-bit (about 5.5 GB incl. embeddings) | 5.5 GB | 14.1 GB | 107,000 | **26** | 53 |

Twenty users at 2,000 tokens need 20 × 2,000 × 128 KiB = 5.2 GB (the worst case, all at 4,096 tokens, is 10.7 GB), so it fits only with the 4-bit weights. Quantisation is what makes 20 users possible.

**Step 3 – speed.** Every decoding step reads the weights once (shared by all 20 sequences) and the KV cache of every sequence: 5.5 GB + 5.2 GB = 10.7 GB. At 60% of peak bandwidth that is 60 ms on an L4, 30 ms on an A10G and 18 ms on an RTX 4090. Each step gives every user one token, so each user gets **about 17, 34 and 56 tokens/s** (total 335, 670 and 1,100 tokens/s). Reading speed is only about 5 to 8 tokens/s, so even the L4 is comfortable. Prefill is compute-bound: a 1,000-token prompt is 2 × 8 × 10⁹ × 1,000 = 16 TFLOP, about 0.4 s; if all 20 prompts arrive in the same second the last user waits about 8 s for the first token (chunked prefill keeps the others streaming).

**Step 4 – how to serve it.** vLLM (continuous batching, so new requests join the running batch at every step; PagedAttention, so KV memory is allocated in small blocks on demand instead of reserving 4,096 tokens per user) with a 4-bit AWQ or GPTQ model: `--max-model-len 4096 --max-num-seqs 20 --gpu-memory-utilization 0.90 --enable-prefix-caching` (the shared system prompt is cached). Cap the answer length (for example 512 tokens), queue requests beyond 20, and watch the KV-cache usage and the 95th-percentile time to first token. If the load grows, use `--kv-cache-dtype fp8` (halves the KV cache), a smaller model, or a second GPU. Ollama with `OLLAMA_NUM_PARALLEL=20` also fits (it reserves 20 × 4,096 tokens = 10.7 GB up front) but usually gives lower throughput than vLLM.